In [11]:
!pip install streamlit pyngrok scikit-learn imbalanced-learn pandas numpy


In [ ]:
import numpy as np
import pandas as pd
import streamlit as st
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
from pyngrok import ngrok
import pickle
import os

# 🚀 Load dataset
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df = df.rename(columns={"Category": "label", "Message": "email"})
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df = df.dropna()

# 🚀 Train model
X_train, X_test, y_train, y_test = train_test_split(df['email'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])
vectorizer = CountVectorizer(max_features=5000)
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts = vectorizer.transform(X_test)

# Handle imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_counts, y_train)

# Train Naïve Bayes Model
model = MultinomialNB()
model.fit(X_train_resampled, y_train_resampled)

# Save trained model and vectorizer
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

# 🚀 Create Streamlit App
with open("app.py", "w") as f:
    f.write("""
import streamlit as st
import pickle
from sklearn.feature_extraction.text import CountVectorizer

# Load trained model and vectorizer
model = pickle.load(open("model.pkl", "rb"))
vectorizer = pickle.load(open("vectorizer.pkl", "rb"))

st.title("📧 Email Spam Detection")

new_email = st.text_area("Enter the email text:")

if st.button("Classify Email"):
    if new_email:
        new_email_counts = vectorizer.transform([new_email])
        pred_prob = model.predict_proba(new_email_counts)
        # Check if pred_prob has the expected shape before accessing elements
        if pred_prob.shape[0] > 0 and pred_prob.shape[1] > 1:
            result = "Spam" if pred_prob[0][1] > 0.4 else "Not Spam"
        else:
            result = "Unable to classify. Please check the input email."  # Handle unexpected shape
        st.write(f"**Prediction:** {result}")
    else:
        st.write("⚠️ Please enter an email to classify.")
""")

# 🚀 Run Streamlit in the background
!streamlit run app.py &

# 🚀 Expose the Streamlit app with ngrok
public_url = ngrok.connect(port=8501)
print(f"🔗 Open your app here: {public_url}")




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.23.124.53:8501

